# Week 11 — Generative AI: redesigning Florida streets with Nano Banana
### From reading the street (Weeks 8–10) to drawing what it could become

In Weeks 8–10 we *read* the Florida images: we clustered them, labelled them and detected objects in them.
Today we **change** them: you give an image model a picture and a design brief in plain English, and it draws the variation.
The model is **Nano Banana**, Google Gemini's image model. Today is hands-on: try, run the cell again, change one word, compare.

```
 1  IMAGINE     text → image: describe a Florida street, get a picture
 2  EDIT        a real street + a design brief → a design variation
 3  REFERENCE   a street + an ArchDaily building → "build something like this here"
 4  YOUR SITE   your own photo → your own variation
 5  BOARD       your best result → a design board (before / after / prompt / what changed)
```

| Section | What you do | Saved in your Drive (`Week11/`) | Time |
|---|---|---|---|
| **0** Setup | mount Drive, settings, class API key | – | 5 min |
| **1** Imagine | text → image, the same street in six cities | `01_imagine/` | 10 min |
| **2** Edit | design briefs on real street views, refine | `02_edit/` | 15 min |
| **3** Reference | a street + an ArchDaily building | `03_reference/` | 10 min |
| **4** Your site | upload your photos and edit them | `my_photos/`, `04_my_site/` | 10 min |
| **5** Board | the design board (PNG + PDF) | `boards/` | 5 min |

**Before you start**
* **No GPU needed**: the images are made on Google's servers (the normal CPU runtime is fine).
* Add the instructor's shared folder **`Week11_shared_data`** to your Drive (open the shared link → *Organize → Add shortcut* → *My Drive*). It holds the example images.
* Cells marked **🍌 make an image** (≈ $0.045 each, paid from one key shared by the whole class). **Running a 🍌 cell again gives a new, different image** — that is the point: try several times.
* There is a daily budget per student (`DAILY_BUDGET_USD` in 0.2). Everything you make is saved in your Drive together with its prompt (`Week11/generations.csv`).

---
# 0 · Setup

In [ ]:
# 0.1  Install, imports, Google Drive and folders
%pip install -q "google-genai>=2.33" pillow-heif
# (a note "you may need to restart the kernel" can be ignored)

import io, os, re, math, time, textwrap, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from google import genai
from google.genai import types, errors

try:
    import pillow_heif
    pillow_heif.register_heif_opener()             # lets PIL open iPhone photos (.heic)
except ImportError:
    pass

%config InlineBackend.figure_formats = ["jpeg"]
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.rcParams.update({"figure.dpi": 100})

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
    COURSE_DIR = MY_DRIVE / "Colab Notebooks" / "ML_for_Architects"
else:  # running on your own computer: the same folders, next to this notebook
    MY_DRIVE = Path.cwd()
    COURSE_DIR = Path.cwd() / "ML_for_Architects"

SHARED_DIR = MY_DRIVE / "Week11_shared_data"       # the instructor's folder (a shortcut in "My Drive")
WEEK11_DIR = COURSE_DIR / "Week11"                 # everything this notebook writes
OUT_DIRS = {s: WEEK11_DIR / s for s in ("01_imagine", "02_edit", "03_reference", "04_my_site")}
MY_PHOTOS_DIR, BOARD_DIR = WEEK11_DIR / "my_photos", WEEK11_DIR / "boards"
LOG_FILE = WEEK11_DIR / "generations.csv"          # one row per image you make
for d in [*OUT_DIRS.values(), MY_PHOTOS_DIR, BOARD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Running on Colab" if IN_COLAB else "Running locally", "| this notebook writes to:", WEEK11_DIR)
print("shared folder:", SHARED_DIR, "(found)" if SHARED_DIR.exists() else "(NOT found -> see 'Before you start')")

### 0.2 Settings
On Colab the settings appear as menus next to the code. Leave them as they are unless your instructor says otherwise.
* `MODEL` — **Nano Banana 2.1** is fast and cheap (≈ $0.045 per image). *Nano Banana 2 Lite* costs about the same but is weaker at editing; *Nano Banana Pro* is the strongest and ≈ 4× more expensive.
* `IMAGE_SIZE` — `1K` (about 1,200 px wide) is enough for the screen and the board; `2K` costs ≈ 50 % more.
* `DAILY_BUDGET_USD` — the most this notebook spends for you per day (the key is shared by the whole class).

In [ ]:
#@title 0.2 Settings
MODEL = "gemini-nano-banana-2.1"   #@param ["gemini-nano-banana-2.1", "gemini-3.1-flash-lite-image", "gemini-3.1-flash-image", "gemini-3-pro-image"]
IMAGE_SIZE = "1K"                  #@param ["1K", "2K"]
DAILY_BUDGET_USD = 1.50            #@param {type:"number"}
YOUR_NAME = "Test"                     #@param {type:"string"}

# estimated $ per image (the image + the model's "thinking"), from the Gemini API price list (October 2026)
PRICE_PER_IMAGE = {"gemini-nano-banana-2.1": {"1K": 0.045, "2K": 0.06},
                   "gemini-3.1-flash-lite-image": {"1K": 0.04, "2K": 0.04},    # Lite makes 1K images only
                   "gemini-3.1-flash-image": {"1K": 0.07, "2K": 0.11},
                   "gemini-3-pro-image": {"1K": 0.16, "2K": 0.16}}
MODEL_NAMES = {"gemini-nano-banana-2.1": "Nano Banana 2.1", "gemini-3.1-flash-lite-image": "Nano Banana 2 Lite",
               "gemini-3.1-flash-image": "Nano Banana 2", "gemini-3-pro-image": "Nano Banana Pro"}
if MODEL == "gemini-3.1-flash-lite-image":
    IMAGE_SIZE = "1K"
print(f"{MODEL_NAMES[MODEL]} ({MODEL}) · {IMAGE_SIZE} · ≈ ${PRICE_PER_IMAGE[MODEL][IMAGE_SIZE]:.3f} per image · "
      f"daily budget ${DAILY_BUDGET_USD:.2f}")

### 0.3 The class API key
The whole class uses one key from your instructor. **Recommended:** store it as a Colab *secret*, so it never appears in the notebook:
click the **🔑 key icon** in the left sidebar → *Add new secret* → name `GEMINI_API_KEY`, value = the key → switch on *Notebook access*.

If your instructor put the key into the shared folder (`Week11_shared_data/class_key.txt`), nothing to do: the notebook finds it.
Pasting the key into `GEMINI_API_KEY` below also works — but then **never share this notebook**.

In [ ]:
# 0.3  API key: pasted value  >  Colab secret "GEMINI_API_KEY"  >  Week11_shared_data/class_key.txt  >  environment variable
GEMINI_API_KEY = "AQ.Ab8RN6LQ1gnm6Bzg62kaS1Imuvs1-JgFwf3gL4Ou7_BuGKldKQ"


def find_api_key():
    if GEMINI_API_KEY.strip():
        return GEMINI_API_KEY.strip(), "pasted in 0.3"
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("GEMINI_API_KEY") or "").strip()
            if key:
                return key, "Colab secret"
        except Exception as e:
            print("No Colab secret available:", type(e).__name__)
    key_file = SHARED_DIR / "class_key.txt"
    if key_file.exists() and key_file.read_text().strip():
        return key_file.read_text().strip(), "Week11_shared_data/class_key.txt"
    return os.environ.get("GEMINI_API_KEY", "").strip(), "environment variable"


API_KEY, KEY_SOURCE = find_api_key()
if not API_KEY:
    raise RuntimeError("No API key found -> add the Colab secret GEMINI_API_KEY (see 0.3), then re-run this cell.")

# when the whole class sends images at the same moment, Google may answer "busy": wait and retry (up to 4 tries)
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(
    timeout=180_000, retry_options=types.HttpRetryOptions(attempts=4, initial_delay=10, max_delay=60,
                                                          http_status_codes=[429, 500, 503, 504])))
try:
    info = client.models.get(model=MODEL)          # free: only asks whether the model is available for this key
except errors.APIError as e:
    raise RuntimeError(f"The key does not work ({e.code}: {e.message}) -> check 0.3 or ask your instructor.") from None
print(f"API key: found ({KEY_SOURCE}, ends with …{API_KEY[-4:]}) | model: {info.display_name} ✓")

### 0.4 Helper functions and example images
Run the cell — you don't need to read it. It defines `generate()` (one call = one image, saved in your Drive and logged in `generations.csv`), the design briefs, and the design board, and it finds the example images: **8 Google Street View images** from Week 8 (one or two per city) and **6 ArchDaily projects** in Florida.

In [ ]:
#@title 0.4 Helpers (run once — you do not need to read this cell)
# ---- example images: in the shared folder, or else in your / the shared Week 8 folder
EXAMPLE_LIST = [  # name, kind, file inside 01_images/<kind>/, credit
    ("Gainesville - houses on a brick street", "streetview", "Gainesville/GNV_downtown_7f99ab.jpg", "Google Street View"),
    ("Jacksonville - parking garage with palms", "streetview", "Jacksonville/JAX_downtown_9acdcb.jpg", "Google Street View"),
    ("Miami - Brickell towers", "streetview", "Miami/MIA_downtown_brickell_c94b58.jpg", "Google Street View"),
    ("Miami - Wynwood warehouse", "streetview", "Miami/MIA_wynwood_861209.jpg", "Google Street View"),
    ("Orlando - vacant lot downtown", "streetview", "Orlando/ORL_downtown_2cb220.jpg", "Google Street View"),
    ("Orlando - wide arterial road", "streetview", "Orlando/ORL_downtown_6d7ce7.jpg", "Google Street View"),
    ("Tallahassee - garage on a campus street", "streetview", "Tallahassee/TLH_collegetown_fsu_8eb903.jpg", "Google Street View"),
    ("Tampa - historic block in Ybor City", "streetview", "Tampa/TPA_ybor_city_6debf6.jpg", "Google Street View"),
    ("Gainesville - UF Malachowsky Hall (folded panels)", "archdaily", "Gainesville/AD_696fa3d09d7ea30001d7eaff.jpg",
     "ArchDaily: Malachowsky Hall, University of Florida / Bohlin Cywinski Jackson"),
    ("Jacksonville - Milam Residence (concrete frames)", "archdaily", "Jacksonville/AD_5037e86928ba0d599b0003d5.jpg",
     "ArchDaily: Milam Residence / Paul Rudolph"),
    ("Miami - Jade Alley (concrete arches)", "archdaily", "Miami/AD_669644cdc8b7c8017cddc07c.jpg",
     "ArchDaily: Jade Alley / Daniel Toole Architecture"),
    ("Orlando - Nemours Children's Hospital (wood fins)", "archdaily", "Orlando/AD_525f55b5e8e44e988d000059.jpg",
     "ArchDaily: Nemours Children's Hospital / Stanley Beaman & Sears + Perkins&Will"),
    ("Orlando - The Landing at Evermore (steel canopy)", "archdaily", "Orlando/AD_6a2b1cc3712cdb0001fbe131.jpg",
     "ArchDaily: The Landing at Evermore Resort / Polk Stanley Wilcox Architects"),
    ("Tampa - Tampa Museum of Art (perforated aluminium)", "archdaily", "Tampa/AD_50089d0128ba0d50da001340.jpg",
     "ArchDaily: Tampa Museum of Art / Stanley Saitowitz | Natoma Architects"),
]
EXAMPLES = {}
for name, kind, rel, credit in EXAMPLE_LIST:
    places = [SHARED_DIR / "examples" / kind / Path(rel).name, COURSE_DIR / "Week8" / "01_images" / kind / rel,
              MY_DRIVE / "Week8_shared_data" / "01_images" / kind / rel]
    path = next((p for p in places if p.exists()), None)
    if path is not None:
        EXAMPLES[name] = dict(kind=kind, path=path, credit=credit,
                              crop=24 if kind == "streetview" else 0)    # street view: cut the Google logo strip
STREET_EXAMPLES = [n for n, e in EXAMPLES.items() if e["kind"] == "streetview"]
ARCH_EXAMPLES = [n for n, e in EXAMPLES.items() if e["kind"] == "archdaily"]

# ---- design briefs (Sections 2 and 4) and reference tasks (Section 3)
BRIEFS = {
    "Shade trees": "Add a continuous row of large native live oak trees along the sidewalks, so that people walk in dappled shade.",
    "Complete street": "Turn one car lane into a protected two-way bike lane with a planted buffer, widen the sidewalk, "
                       "and add benches, bike racks and a bus shelter.",
    "King-tide flood": "Show this exact scene during a king-tide flood: about 30 cm of sea water covers the street and "
                       "reflects the buildings; overcast sky.",
    "Flood-ready street": "Make the street flood-ready: raised sidewalks, rain gardens with native plants along the curb, "
                          "permeable paving, and floodproof ground floors.",
    "Hurricane-ready": "Make the street hurricane-ready: remove the overhead power lines and poles, add impact-resistant "
                       "storm shutters to the windows, and plant sturdy native sabal palms.",
    "More density": "Replace the low buildings with four-storey mixed-use buildings with shops and a shaded arcade on "
                    "the ground floor; keep the street.",
    "Adaptive reuse": "Convert the ground floor of the main building into a café and food market with large openings, "
                      "awnings and outdoor seating; keep the structure.",
    "Night market": "Turn the street into a car-free night market at dusk: food stalls, string lights, people walking "
                    "and sitting at tables.",
    "Mural and cool roof": "Paint a large, colourful mural on the main wall and give the roofs a white cool-roof finish.",
}
KEEP_SUFFIX = " Change only what is asked; keep the camera position, the perspective and the light the same. Photorealistic."
REFERENCE_TASKS = {
    "Build it here": "Design a new building for the site in image 1 in the architectural language of the building in "
                     "image 2: its materials, facade and proportions. Put it where it fits best (an empty lot, a parking "
                     "area or the most prominent building). Keep the street, the camera position and the light of image 1. "
                     "Photorealistic.",
    "New facade": "Give the main building in image 1 a new facade inspired by the building in image 2: its materials, "
                  "pattern and colours. Keep the building's volume, the street, the camera position and the light of "
                  "image 1. Photorealistic.",
    "Pavilion": "Add a small public pavilion or shade structure inspired by the building in image 2 to the sidewalk or "
                "open space in image 1. Keep everything else in image 1. Photorealistic.",
}


def brief_prompt(brief, extra="", keep=True):
    """The text sent to the model: the chosen brief (+ your extra words) (+ 'keep the rest')."""
    extra = extra.strip()
    if brief.startswith("My own"):
        if not extra:
            raise ValueError("Write your brief in EXTRA.")
        text = extra
    else:
        text = BRIEFS[brief] + (" " + extra if extra else "")
    return text + (KEEP_SUFFIX if keep else "")


# ---- the log of everything you make
LOG_COLS = ["gen_id", "time", "section", "model", "image_size", "aspect_ratio", "prompt", "inputs", "output", "status",
            "seconds", "est_usd", "prompt_tokens", "output_tokens", "thinking_tokens"]


def load_log():
    if LOG_FILE.exists():
        return pd.read_csv(LOG_FILE, dtype=str, keep_default_na=False)
    return pd.DataFrame(columns=LOG_COLS)


def gen_row(gen_id):
    log = load_log()
    hit = log[(log.gen_id == str(gen_id).lstrip("#").zfill(3)) & (log.status == "ok")]
    return None if hit.empty else hit.iloc[-1]


def spent_today():
    log = load_log()
    ok = log[(log.status == "ok") & log.time.str.startswith(datetime.now().strftime("%Y-%m-%d"))]
    return float(pd.to_numeric(ok.est_usd, errors="coerce").sum())


# ---- image sources: "example:<name>", "gen:<number>", "photo:<file name>" or "latest"
PHOTO_EXTS = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"}


def list_photos():
    return sorted(p for p in MY_PHOTOS_DIR.iterdir() if p.suffix.lower() in PHOTO_EXTS)


def resolve(src):
    if src == "latest":
        ok = load_log().query("status == 'ok'")
        if ok.empty:
            raise ValueError("No image yet -> make one first.")
        return f"gen:{ok.gen_id.iloc[-1]}"
    if src.startswith("gen:"):
        return "gen:" + src[4:].strip().lstrip("#").zfill(3)
    return src


def section_of(src, default="02_edit"):
    """The folder for a refinement: the folder of the image it starts from (text-only images -> 02_edit)."""
    try:
        src = resolve(src)
    except ValueError:
        return default
    row = gen_row(src[4:]) if src.startswith("gen:") else None
    return row["section"] if row is not None and row["section"] != "01_imagine" else default


def input_image(src):
    kind, _, value = resolve(src).partition(":")
    if kind == "example":
        if value not in EXAMPLES:
            raise ValueError(f"Example image not found: {value} -> add the shared folder (see 'Before you start').")
        e = EXAMPLES[value]
        img = Image.open(e["path"]).convert("RGB")
        return img.crop((0, 0, img.width, img.height - e["crop"])) if e["crop"] else img
    if kind == "gen":
        row = gen_row(value)
        if row is None:
            raise ValueError(f"No image #{value} in generations.csv")
        return Image.open(WEEK11_DIR / row["output"]).convert("RGB")
    if kind == "photo":
        img = ImageOps.exif_transpose(Image.open(MY_PHOTOS_DIR / value)).convert("RGB")
        img.thumbnail((1536, 1536))                # phone photos are large: 1536 px is plenty and uploads faster
        return img
    raise ValueError(f"Unknown image source: {src}")


def source_name(src):
    kind, _, value = src.partition(":")
    return {"example": value, "gen": f"#{value}", "photo": f"my photo {value}"}.get(kind, src)


def source_credits(src, depth=0):
    kind, _, value = src.partition(":")
    if kind == "example":
        return [EXAMPLES[value]["credit"]] if value in EXAMPLES else []
    if kind == "photo":
        return [f"photo: {YOUR_NAME}" if YOUR_NAME else "own photo"]
    row = gen_row(value) if kind == "gen" and depth < 10 else None
    return [] if row is None else [c for s in row["inputs"].split(" | ") if s for c in source_credits(s, depth + 1)]


def show_images(images, titles=None, height=4.0):
    images = list(images)
    widths = [im.width / im.height for im in images]
    height = min(height, 17 / sum(widths))
    fig, axes = plt.subplots(1, len(images), figsize=(height * sum(widths) + 0.2 * len(images), height + 0.5),
                             gridspec_kw={"width_ratios": widths}, squeeze=False)
    for ax, im, t in zip(axes[0], images, titles or [""] * len(images)):
        ax.imshow(im)
        ax.set_title(textwrap.shorten(t, 55), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def slug(text, n=40):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:n].strip("-")


# ---- generate(): one call = one image
def explain_error(e):
    msg = f"{e.code}: {e.message}"
    if e.code == 402:
        return "the class credit is used up -> tell your instructor. (" + msg + ")"
    if e.code == 429:
        return "too many requests from the class right now -> wait a minute and run the cell again. (" + msg + ")"
    if e.code in (500, 503, 504):
        return "Google's servers are busy -> run the cell again. (" + msg + ")"
    return msg


def generate(prompt, inputs=(), section="01_imagine", aspect_ratio=None, show=True):
    """Make ONE image from a prompt (+ input images). Saves it, logs it, shows it. Returns its number, e.g. '014'."""
    price = PRICE_PER_IMAGE[MODEL][IMAGE_SIZE]
    spent = spent_today()
    if spent + price > DAILY_BUDGET_USD + 1e-9:
        print(f"⛔ Daily budget reached: ${spent:.2f} of ${DAILY_BUDGET_USD:.2f} used today -> work with the images "
              "you have (Section 5), or ask your instructor.")
        return None
    try:
        inputs = [resolve(s) for s in inputs]
        images = [input_image(s) for s in inputs]
    except (ValueError, OSError) as e:
        print("⚠️ ", e)
        return None
    contents = []
    for k, img in enumerate(images, 1):
        if len(images) > 1:
            contents.append(f"Image {k}:")
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=92)
        contents.append(types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"))
    contents.append(prompt)
    image_config = types.ImageConfig(image_size=IMAGE_SIZE, aspect_ratio=aspect_ratio)  # no ratio: same as the input
    config = types.GenerateContentConfig(response_modalities=["IMAGE"], image_config=image_config)

    log = load_log()
    gen_id = f"{len(log) + 1:03d}"
    row = dict.fromkeys(LOG_COLS, "")
    row.update(gen_id=gen_id, time=datetime.now().isoformat(timespec="seconds"), section=section, model=MODEL,
               image_size=IMAGE_SIZE, aspect_ratio=aspect_ratio or "", prompt=prompt, inputs=" | ".join(inputs),
               est_usd="0")
    print(f"🍌 #{gen_id} · {MODEL_NAMES[MODEL]} is drawing … (10–30 s)")
    t0, data, problem = time.time(), None, None
    try:
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        cand = response.candidates[0] if response.candidates else None
        for part in (cand.content.parts or []) if cand and cand.content else []:
            if part.inline_data and part.inline_data.data and not part.thought:
                data, mime = part.inline_data.data, part.inline_data.mime_type
        if data is None:
            reason = getattr(cand, "finish_reason", None) if cand else None
            block = getattr(response.prompt_feedback, "block_reason", None) if response.prompt_feedback else None
            why = getattr(reason or block, "name", None) or str(reason or block or "no image")
            problem = (f"no image (reason: {why}). The model refused or failed -> change the prompt "
                       "(no real people's names, no brands, nothing dangerous) and run again.")
        u = response.usage_metadata
        if u is not None:
            row.update(prompt_tokens=u.prompt_token_count or "", output_tokens=u.candidates_token_count or "",
                       thinking_tokens=u.thoughts_token_count or "")
    except errors.APIError as e:
        problem = explain_error(e)
    row["seconds"] = f"{time.time() - t0:.0f}"
    if data is not None:
        out = OUT_DIRS[section] / f"{gen_id}_{slug(prompt)}{'.png' if mime == 'image/png' else '.jpg'}"
        out.write_bytes(data)
        row.update(status="ok", output=out.relative_to(WEEK11_DIR).as_posix(), est_usd=f"{price:.3f}")
    else:
        row["status"] = "failed: " + problem
    pd.concat([log, pd.DataFrame([row])], ignore_index=True).to_csv(LOG_FILE, index=False)
    if data is None:
        print(f"⚠️  #{gen_id}: {problem}")
        return None
    print(f"✓ #{gen_id} · {row['seconds']} s · ≈ ${price:.3f} · today ${spent + price:.2f} of ${DAILY_BUDGET_USD:.2f} "
          f"· saved in Week11/{row['output']}")
    if show:
        result = Image.open(io.BytesIO(data))
        names = [("before: " if len(inputs) == 1 else f"image {k}: ") + source_name(s) for k, s in enumerate(inputs, 1)]
        show_images([*images, result], names + [f"#{gen_id} · {MODEL_NAMES[MODEL]}"])
    return gen_id


def gallery(n=12, cols=4):
    """Your last n images with their numbers."""
    ok = load_log().query("status == 'ok'").tail(n)
    if ok.empty:
        print("No images yet.")
        return
    nrows = math.ceil(len(ok) / cols)
    fig, axes = plt.subplots(nrows, cols, figsize=(4.3 * cols, 3.5 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, r in zip(axes.ravel(), ok.itertuples()):
        im = Image.open(WEEK11_DIR / r.output)
        im.thumbnail((640, 640))
        ax.imshow(im)
        ax.set_title(f"#{r.gen_id} · {textwrap.shorten(r.prompt, 38)}", fontsize=9)
    plt.tight_layout()
    plt.show()


# ---- the design board: before → after, prompt, and what changed (measured as in Week 8)
SEG_GROUPS = {  # SegFormer (ADE20K) classes grouped as in Week 8
    "sky": ["sky"],
    "buildings & walls": ["building", "house", "skyscraper", "tower", "hovel", "windowpane", "door", "column", "awning",
                          "canopy", "wall"],
    "trees & plants": ["tree", "grass", "plant", "palm", "flower", "field"],
    "road": ["road", "path", "runway", "dirt track"],
    "sidewalk": ["sidewalk", "stairs", "step", "stairway"],
    "vehicles": ["car", "bus", "truck", "van", "minibike", "bicycle", "boat", "ship", "airplane"],
    "people": ["person"],
    "water": ["water", "sea", "river", "lake", "swimming pool", "fountain"],
}
_SEG = {}


def measure(img):
    """Share of the picture covered by sky, buildings, trees, … (SegFormer-b0, the model of Week 8)."""
    if not _SEG:            # first call: download the small model (15 MB); a warning about HF_TOKEN can be ignored
        import torch
        from transformers import SegformerForSemanticSegmentation
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
        hf_logging.disable_progress_bar()
        model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512").eval()
        labels = [model.config.id2label[i].strip() for i in range(model.config.num_labels)]
        _SEG.update(torch=torch, model=model,
                    idx={g: [labels.index(n) for n in names if n in labels] for g, names in SEG_GROUPS.items()})
    x = np.asarray(img.convert("RGB").resize((512, 512), Image.BILINEAR), dtype=np.float32) / 255
    x = ((x - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])).astype(np.float32)
    with _SEG["torch"].inference_mode():
        logits = _SEG["model"](pixel_values=_SEG["torch"].from_numpy(x.transpose(2, 0, 1)[None].copy())).logits
    labelmap = logits.argmax(1)[0].numpy()
    frac = np.bincount(labelmap.ravel(), minlength=logits.shape[1]) / labelmap.size
    return {g: float(frac[i].sum()) for g, i in _SEG["idx"].items()}


def make_board(selection="latest", title="", note="", measure_changes=True):
    """Design board for up to 3 images: 'latest' or numbers such as '014, 017'. Saved as PNG + PDF in boards/."""
    ok = load_log().query("status == 'ok'")
    if ok.empty:
        print("No images yet -> make one in Sections 1–4 first.")
        return None
    if selection.strip().lower() in ("", "latest"):
        ids = [ok.gen_id.iloc[-1]]
    else:
        ids = [s.lstrip("#").zfill(3) for s in re.split(r"[,;\s]+", selection) if s.strip()][:3]
    rows = [gen_row(i) for i in ids]
    if any(r is None for r in rows):
        print("Unknown image number(s):", [i for i, r in zip(ids, rows) if r is None], "-> see the gallery (5.1)")
        return None
    # the picture row: before → after(s); a second input image is the reference
    selected, panels = {f"gen:{r['gen_id']}" for r in rows}, []
    for r in rows:
        for k, s in enumerate([s for s in r["inputs"].split(" | ") if s]):
            if s not in selected and s not in [p[1] for p in panels]:
                panels.append(("reference" if k else "before", s))
        panels.append(("after", f"gen:{r['gen_id']}"))
    images = [input_image(s) for _, s in panels]
    befores = [s for kind, s in panels if kind == "before"]
    short = {s: ("before" if len(befores) == 1 else f"before {befores.index(s) + 1}") if kind == "before"
             else "reference" if kind == "reference" else f"#{s[4:]}" for kind, s in panels}
    labels = [f"{short[s]} · {source_name(s)}" if kind != "after" else f"#{s[4:]} · after" for kind, s in panels]

    widths = [im.width / im.height for im in images]
    row_h, text_h = min(5.5, 15.2 / sum(widths)), 4.4             # inches: picture row, text + chart
    fig_h = 1.35 + row_h + 0.7 + text_h
    fig = plt.figure(figsize=(16, fig_h))
    outer = fig.add_gridspec(2, 1, height_ratios=[row_h, text_h], hspace=0.7 / ((row_h + text_h) / 2),
                             top=1 - 1.35 / fig_h, bottom=0.15 / fig_h, left=0.03, right=0.97)
    top = outer[0].subgridspec(1, len(images), width_ratios=widths, wspace=0.03)
    for k, (im, lab, (kind, _)) in enumerate(zip(images, labels, panels)):
        ax = fig.add_subplot(top[0, k])
        ax.imshow(im)
        ax.axis("off")
        ax.set_title(textwrap.shorten(lab, 42), fontsize=11, color="#666666" if kind != "after" else "black",
                     weight="bold" if kind == "after" else "normal")
    fig.text(0.03, 1 - 0.15 / fig_h, title or "Design variation", fontsize=22, weight="bold", va="top")
    fig.text(0.03, 1 - 0.62 / fig_h, f"{YOUR_NAME + ' · ' if YOUR_NAME else ''}ML for Architects · Week 11 · "
                                     f"{datetime.now():%d %B %Y}", fontsize=11, color="#555555", va="top")

    bottom = outer[1].subgridspec(1, 2, width_ratios=[1, 1] if measure_changes else [1, 0.001], wspace=0.32)
    lines = []
    for r in rows:
        lines += [f"#{r['gen_id']}  " + textwrap.fill(textwrap.shorten(r["prompt"], 400), 78,
                                                      subsequent_indent="        "), ""]
    if note.strip():
        lines += [textwrap.fill("Note: " + note.strip(), 80), ""]
    credits = sorted({c for r in rows for s in r["inputs"].split(" | ") if s for c in source_credits(s)})
    models = sorted({f"{MODEL_NAMES.get(r['model'], r['model'])} ({r['model']}, {r['image_size']})" for r in rows})
    lines += [textwrap.fill("Generated with " + ", ".join(models) + ". AI-generated images (invisible SynthID "
                            "watermark).", 80),
              textwrap.fill("Images: " + ("; ".join(credits) if credits else "text only"), 80)]
    axt = fig.add_subplot(bottom[0, 0])
    axt.axis("off")
    axt.text(0, 1, "\n".join(lines), va="top", ha="left", fontsize=10, linespacing=1.35)

    if measure_changes:
        axc = fig.add_subplot(bottom[0, 1])
        try:
            series = [(short[s], kind, measure(im)) for im, (kind, s) in zip(images, panels) if kind != "reference"]
            groups = list(SEG_GROUPS)[::-1]
            greys, colours = iter(["#9e9e9e", "#cfcfcf", "#6e6e6e"]), iter(["#2e7d32", "#1565c0", "#ef6c00"])
            h = 0.8 / len(series)
            for j, (name, kind, shares) in enumerate(series):
                y = np.arange(len(groups)) + 0.4 - h * (j + 0.5)
                vals = [100 * shares[g] for g in groups]
                axc.barh(y, vals, height=h, color=next(greys if kind == "before" else colours), label=name)
                for yy, v in zip(y, vals):
                    if v >= 1:
                        axc.text(v + 0.5, yy, f"{v:.0f}", va="center", fontsize=7, color="#444444")
            axc.set_yticks(np.arange(len(groups)))
            axc.set_yticklabels(groups)
            axc.set_xlabel("share of the picture (%)")
            axc.set_title("What changed? (measured with SegFormer, as in Week 8)", fontsize=11)
            axc.spines[["top", "right"]].set_visible(False)
            axc.legend(fontsize=9, loc="lower right", frameon=False)
        except Exception as e:                       # e.g. no internet for the model download
            axc.axis("off")
            axc.text(0, 0.5, f"(measurement skipped: {type(e).__name__})", fontsize=10)
    stem = BOARD_DIR / f"board_{'_'.join(ids)}"
    fig.savefig(stem.with_suffix(".png"), dpi=150, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    print(f"saved: Week11/boards/{stem.name}.png and .pdf")
    return stem.with_suffix(".png")


print(f"example images: {len(STREET_EXAMPLES)} street views, {len(ARCH_EXAMPLES)} ArchDaily"
      + ("" if EXAMPLES else "  -> NOT found: add the shared folder Week11_shared_data (see 'Before you start')"))
print(f"images made so far: {(load_log().status == 'ok').sum()} | spent today ≈ ${spent_today():.2f} of ${DAILY_BUDGET_USD:.2f}")

---
# 1 · Imagine: text → image
Write what you want to see and run the cell. **Run it again for a new version** — each run is a new drawing (≈ $0.045).

In [ ]:
#@title 1.1 🍌 Your first image
PROMPT = "A typical street in Miami"   #@param {type:"string"}
ASPECT_RATIO = "4:3"                   #@param ["4:3", "3:2", "16:9", "1:1", "3:4", "9:16"]
gid = generate(PROMPT, section="01_imagine", aspect_ratio=ASPECT_RATIO)

✏️ **Try:** run 1.1 two or three times — what changes, what stays the same? Then be more precise, e.g. *"A residential street in Little Havana, Miami, at noon, seen from the sidewalk"*.

### 1.2 A prompt recipe
A good prompt says **what**, **where**, **when**, **how it is seen** and **in what style**. Change one field at a time and run again.

In [ ]:
#@title 1.2 🍌 Prompt recipe
WHAT = "a shaded two-lane street with small shops on the ground floor and people walking"   #@param {type:"string"}
WHERE = "in Ybor City, Tampa, Florida"                                                       #@param {type:"string"}
WHEN = "late afternoon after a summer rain, wet pavement reflecting the sky"                 #@param {type:"string"}
VIEW = "eye-level view from the sidewalk, 35 mm lens"                                        #@param {type:"string"}
STYLE = "realistic photograph"   #@param ["realistic photograph", "architectural watercolour", "pencil sketch", "isometric diagram", "aerial drone photo", "1950s Florida postcard", "night photo with neon lights"]

prompt = f"{STYLE[0].upper() + STYLE[1:]} of {WHAT}, {WHERE}. {WHEN[0].upper() + WHEN[1:]}. {VIEW[0].upper() + VIEW[1:]}."
print(prompt)
gid = generate(prompt, section="01_imagine", aspect_ratio="4:3")

### 1.3 The same street in six cities (6 images, ≈ $0.27)
How does the model imagine each Florida city? **Top:** the model's picture. **Bottom:** a real street view of the same city (Week 8).

In [ ]:
#@title 1.3 🍌 Six cities
STREET_TYPE = "a typical downtown street"   #@param {type:"string"}
CITIES = ["Gainesville", "Tallahassee", "Jacksonville", "Orlando", "Tampa", "Miami"]

made = {city: generate(f"Realistic eye-level photo of {STREET_TYPE} in {city}, Florida.", section="01_imagine",
                       aspect_ratio="4:3", show=False) for city in CITIES}
fig, axes = plt.subplots(2, len(CITIES), figsize=(20, 6.2))
for k, city in enumerate(CITIES):
    real = next((n for n in STREET_EXAMPLES if n.startswith(city)), None)
    for r, (src, what) in enumerate([(f"gen:{made[city]}" if made[city] else None, "model"),
                                     (f"example:{real}" if real else None, "real (Week 8)")]):
        axes[r, k].axis("off")
        if src:
            axes[r, k].imshow(input_image(src))
        axes[r, k].set_title(f"{city} · {what}", fontsize=10)
plt.tight_layout()
plt.show()

✏️ **Discuss:** Which differences between the six cities are real, and which are clichés (palm trees everywhere? pastel colours? always sunny)? What would the model need to know to draw *your* street?

---
# 2 · Edit: design variations of a real street
Start from a real street (Week 8), add a **design brief**, get the variation. The model keeps what you don't mention — mostly. Watch what it changes anyway.

In [ ]:
# 2.1 The example streets (Google Street View, Week 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax in axes.ravel():
    ax.axis("off")
for ax, name in zip(axes.ravel(), STREET_EXAMPLES):
    ax.imshow(input_image(f"example:{name}"))
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

### 2.2 Street + design brief
Pick a street and a brief (the brief texts are in `BRIEFS`, cell 0.4 — the exact text sent is printed). Add your own words in `EXTRA`, or choose *My own brief* and write the whole brief there.
`KEEP_THE_REST` adds: *"Change only what is asked; keep the camera position, the perspective and the light the same."*

In [ ]:
#@title 2.2 🍌 Street + design brief
STREET = "Tampa - historic block in Ybor City"   #@param ["Gainesville - houses on a brick street", "Jacksonville - parking garage with palms", "Miami - Brickell towers", "Miami - Wynwood warehouse", "Orlando - vacant lot downtown", "Orlando - wide arterial road", "Tallahassee - garage on a campus street", "Tampa - historic block in Ybor City"]
BRIEF = "Shade trees"   #@param ["Shade trees", "Complete street", "King-tide flood", "Flood-ready street", "Hurricane-ready", "More density", "Adaptive reuse", "Night market", "Mural and cool roof", "My own brief (write it in EXTRA)"]
EXTRA = ""              #@param {type:"string"}
KEEP_THE_REST = True    #@param {type:"boolean"}

prompt = brief_prompt(BRIEF, EXTRA, KEEP_THE_REST)
print(prompt)
gid = generate(prompt, inputs=[f"example:{STREET}"], section="02_edit")

✏️ **Try:** the same brief on two other streets · the same street with three briefs · *King-tide flood* first, then *Flood-ready street* on the same street.

### 2.3 Refine: one change at a time
Not quite right? Give **one** more instruction. The cell starts from your **latest** image — or from any number (e.g. `014`, see `generations.csv` or the gallery in 5.1) — so you can run it again and again, like a design review.

In [ ]:
#@title 2.3 🍌 Refine
CHANGE = "Make the trees a little smaller and add a few people walking in the shade"   #@param {type:"string"}
START_FROM = "latest"   #@param {type:"string"}

src = "latest" if START_FROM.strip().lower() in ("", "latest") else f"gen:{START_FROM.strip()}"
gid = generate(CHANGE + KEEP_SUFFIX, inputs=[src], section=section_of(src))

✏️ **Discuss:** Compare before and after. What did the model change that you did **not** ask for — the buildings, the cars, the signs, the sky? (The board in Section 5 measures it.)

---
# 3 · Reference: a street + an ArchDaily building
Give the model **two images** — the site (image 1) and a reference building from ArchDaily (image 2) — and ask it to build something *like* the reference *on* the site.

In [ ]:
# 3.1 The reference buildings (ArchDaily, Week 8)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax in axes.ravel():
    ax.axis("off")
for ax, name in zip(axes.ravel(), ARCH_EXAMPLES):
    ax.imshow(input_image(f"example:{name}"))
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
#@title 3.2 🍌 Site + reference
SITE = "my latest image"   #@param ["Gainesville - houses on a brick street", "Jacksonville - parking garage with palms", "Miami - Brickell towers", "Miami - Wynwood warehouse", "Orlando - vacant lot downtown", "Orlando - wide arterial road", "Tallahassee - garage on a campus street", "Tampa - historic block in Ybor City", "my latest image", "my first photo (Section 4)"]
REFERENCE = "Miami - Jade Alley (concrete arches)"   #@param ["Gainesville - UF Malachowsky Hall (folded panels)", "Jacksonville - Milam Residence (concrete frames)", "Miami - Jade Alley (concrete arches)", "Orlando - Nemours Children's Hospital (wood fins)", "Orlando - The Landing at Evermore (steel canopy)", "Tampa - Tampa Museum of Art (perforated aluminium)"]
TASK = "Build it here"   #@param ["Build it here", "New facade", "Pavilion"]
EXTRA = ""               #@param {type:"string"}

if SITE == "my latest image":
    site = "latest"
elif SITE.startswith("my first photo"):
    site = f"photo:{list_photos()[0].name}" if list_photos() else f"example:{STREET_EXAMPLES[0]}"
    if not list_photos():
        print("No photo in Week11/my_photos/ yet (Section 4) -> using an example street:", STREET_EXAMPLES[0])
else:
    site = f"example:{SITE}"
prompt = REFERENCE_TASKS[TASK] + (" " + EXTRA.strip() if EXTRA.strip() else "")
print(prompt)
gid = generate(prompt, inputs=[site, f"example:{REFERENCE}"], section="03_reference")

✏️ **Try:** the same site with three references · *New facade* on the Wynwood warehouse with the Tampa Museum of Art · *Pavilion* in front of the Jacksonville garage.

---
# 4 · Your site: your own photos
Upload 1–3 photos of a street or building you know (phone photos are fine, also iPhone `.heic`). They are saved in your Drive in `Week11/my_photos/`.
No photo with you? Set `UPLOAD = False` and run the cell: 4.2 then uses an example street.

In [ ]:
#@title 4.1 Upload your photos (click "Choose files" below the cell)
UPLOAD = True   #@param {type:"boolean"}
if UPLOAD and IN_COLAB:
    from google.colab import files
    for fname, data in files.upload().items():
        (MY_PHOTOS_DIR / fname).write_bytes(data)

photos = list_photos()
print(f"{len(photos)} photo(s) in Week11/my_photos/:", [p.name for p in photos])
if photos:
    show_images([input_image(f"photo:{p.name}") for p in photos[:6]],
                [f"PHOTO_NUMBER = {k}: {p.name}" for k, p in enumerate(photos[:6], 1)], height=3)

In [ ]:
#@title 4.2 🍌 Your photo + design brief
PHOTO_NUMBER = 1        #@param {type:"integer"}
BRIEF = "Shade trees"   #@param ["Shade trees", "Complete street", "King-tide flood", "Flood-ready street", "Hurricane-ready", "More density", "Adaptive reuse", "Night market", "Mural and cool roof", "My own brief (write it in EXTRA)"]
EXTRA = ""              #@param {type:"string"}
KEEP_THE_REST = True    #@param {type:"boolean"}

photos = list_photos()
if photos:
    src = f"photo:{photos[min(max(PHOTO_NUMBER, 1), len(photos)) - 1].name}"
else:
    src = f"example:{STREET_EXAMPLES[0]}"
    print("No photo in Week11/my_photos/ -> using an example street:", STREET_EXAMPLES[0])
prompt = brief_prompt(BRIEF, EXTRA, KEEP_THE_REST)
print(prompt)
gid = generate(prompt, inputs=[src], section="04_my_site")

✏️ **Try:** refine your result with **2.3** (it starts from your latest image), or use your photo as the site in **3.2** (`SITE = "my first photo (Section 4)"`).

---
# 5 · Design board
Pick your best result. The board shows **before → after**, the prompt, and **what changed**: the share of sky, buildings, trees, road … in each picture, measured with the same segmentation model as in Week 8 (SegFormer).

In [ ]:
# 5.1 Your latest images and their numbers
gallery(12)

`BOARD_IMAGES` = `latest`, or up to three numbers from the gallery, e.g. `014, 015, 016` (three briefs on the same street) or `014, 018, 021` (a sequence of refinements).

In [ ]:
#@title 5.2 Make the board
BOARD_IMAGES = "latest"            #@param {type:"string"}
TITLE = "Shade for Ybor City"      #@param {type:"string"}
NOTE = ""                          #@param {type:"string"}
MEASURE = True                     #@param {type:"boolean"}
board = make_board(BOARD_IMAGES, TITLE, NOTE, MEASURE)

---
# 6 · Wrap-up: what is now in your Drive?

In [ ]:
# 6.1 Checklist of the outputs
log = load_log()
ok = log[log.status == "ok"]
rows = [{"folder": f"Week11/{s}/", "images": len(list(d.glob("*.*")))} for s, d in OUT_DIRS.items()]
rows += [{"folder": "Week11/my_photos/", "images": len(list_photos())},
         {"folder": "Week11/boards/", "images": len(list(BOARD_DIR.glob("*.png")))}]
display(pd.DataFrame(rows).set_index("folder"))
print(f"{len(ok)} images made ({len(log) - len(ok)} attempts without an image) · estimated cost "
      f"≈ ${pd.to_numeric(ok.est_usd, errors='coerce').sum():.2f} (today ${spent_today():.2f})")
print("boards:", sorted(p.name for p in BOARD_DIR.glob("*.png")))

## What you have now
| In `Week11/` | Content |
|---|---|
| `01_imagine/` · `02_edit/` · `03_reference/` · `04_my_site/` | every image you made, numbered (`014_….jpg`) |
| `generations.csv` | one row per image: prompt, input images, model, time, estimated cost (and the attempts that failed) |
| `my_photos/` | your uploaded photos |
| `boards/` | your design boards (`.png` for slides, `.pdf` for print) |

All images made by the model carry an invisible **SynthID** watermark that marks them as AI-generated. When you show them, say so — the board does it for you.

## Exercises (in class if time is left, or as homework — pick two)
1. **One brief, three streets.** Run the same brief (2.2) on three example streets, then put the three results on one board (`BOARD_IMAGES = "…, …, …"`). Does the brief work equally well everywhere?
2. **Three models.** Make the same edit with `gemini-3.1-flash-lite-image`, `gemini-nano-banana-2.1` and `gemini-3-pro-image` (0.2 — Pro costs ≈ $0.16 per image). Which one keeps the street most faithfully? Compare the *what changed* bars.
3. **Phasing.** Start from one street and refine twice (2.3): *today → 2030 (shade, bike lane) → 2050 (flood-ready, denser)*. Put the steps on one board.
4. **Your site.** Three briefs on your own photo. Choose the best and explain in `NOTE` (5.2), in two sentences, why.